# Brasa: run the MODIS → VIIRS model (U-Net + MLP ensemble)

Produces VIIRS S-NPP-equivalent hotspot counts per 0.25° cell and month **from MODIS alone**, for July 2002 – 2025, including the pre-VIIRS years 2002–2011.
The prediction is the mean of the trained U-Net (`model/unet.pt`) and a small per-pixel MLP. On held-out 2019–2024 it beats Brasa M3 every year: median error at 5° is 0.203, against 0.246 for Brasa M3, and the NOAA-20 ceiling is 0.105.

| Step | What | Time (T4) |
|---|---|---|
| 1 | Build the MODIS input tensors for 2002–2025 | ~15 min, first run only |
| 2 | Train the MLP half (the U-Net is already trained) | ~8 min, first run only: saved to Drive |
| 3 | Predict every month and write results | ~10 min |

**Needs on Drive** (`MyDrive/brasa/`): `brasa-colab.tar`, and the three updated scripts in `MyDrive/brasa/code/`.
*Runtime → Change runtime type → T4 GPU*, then *Run all*. Outputs go to **`MyDrive/brasa/output/`**.

**Known limits:** in Brazil the ensemble is less accurate than Brasa M3 (country error 0.078 vs 0.054), and the Terra-only months before July 2002 are not covered. Keep Brasa M3 for both.

In [ ]:
# --- settings ---------------------------------------------------------------
BRASA = "/content/drive/MyDrive/brasa"
BUNDLE = f"{BRASA}/brasa-colab.tar"
MODELS = f"{BRASA}/models"          # MLP weights are saved here after the first run
OUT = f"{BRASA}/output"             # results
WORK = "/content/work"
START, END = "2002-07", "2025-12"   # Aqua MODIS starts July 2002; earlier months are Terra-only

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> GPU"

In [ ]:
from google.colab import drive, files
drive.mount("/content/drive")
import os, shutil, glob
if not os.path.exists(BUNDLE):
    print("brasa-colab.tar not found on Drive: pick it in the upload dialog")
    up = files.upload(); os.makedirs(BRASA, exist_ok=True); shutil.move(next(iter(up)), BUNDLE)
for d in (MODELS, OUT, WORK):
    os.makedirs(d, exist_ok=True)
if not os.path.exists(f"{WORK}/brasa/ml/train.py"):
    !tar -xf "{BUNDLE}" -C "{WORK}"
REPO, DER = f"{WORK}/brasa", f"{WORK}/firms/derived"
STACK = f"{DER}/ml/full"
# newer scripts than the ones inside the bundle
for f in glob.glob(f"{BRASA}/code/*.py"):
    shutil.copy(f, f"{REPO}/ml/"); print("updated", os.path.basename(f))
assert "context_features" in open(f"{REPO}/ml/core.py").read(), f"put the updated core.py in {BRASA}/code/"
%cd {REPO}
!pip -q install duckdb

## Step 1 – MODIS input tensors, 2002–2025 (~15 min, CPU, ≈7.5 GB local disk)

In [ ]:
if not os.path.exists(f"{STACK}/meta.json"):
    !python -I ml/build_tensor.py {DER} --years 2002-2025 --name full
else:
    print("tensors already built")

## Step 2 – MLP weights (trained once, then reused from Drive)

In [ ]:
CTX = f"{MODELS}/ctx_models.pt"
if not os.path.exists(CTX):
    !python -I ml/context_baseline.py {STACK} {WORK}/ctx --no-predict 2>&1 | tee {OUT}/train_mlp.log
    shutil.copy(f"{WORK}/ctx/ctx_models.pt", CTX)
else:
    print("using saved", CTX)

## Step 3 – predict every month (~10 min)

In [ ]:
!python -I ml/predict_ensemble.py {STACK} {REPO}/model/unet.pt {CTX} {WORK}/out --start {START} --end {END} --countries {DER}/app/cell_country.parquet
for f in glob.glob(f"{WORK}/out/*"):
    shutil.copy(f, OUT)
!ls -la {OUT}

## Quick look
`pred` is the ensemble (use this), `unet` and `mlp` are its two halves, `viirs_obs` is observed S-NPP from 2012 on (partial in VIIRS outage months), and `modis × 4.1` is the naive global ratio.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
t = pd.read_csv(f"{OUT}/monthly_totals.csv")
g = t[t.region == "GLOBAL"].groupby("year")[["modis", "unet", "mlp", "pred", "viirs_obs"]].sum(min_count=1)
g.insert(1, "modis x 4.1", 4.1 * g.modis)
print((g / 1e6).round(2).to_string())   # millions of detections per year
ax = (g[["modis x 4.1", "pred", "viirs_obs"]] / 1e6).plot(marker="o", figsize=(10, 4))
ax.set_ylabel("detections per year (millions)"); ax.set_title("Global VIIRS-equivalent fire detections")
plt.show()

In [ ]:
# one country's monthly series (Brazil: Brasa M3 is still more accurate here, see the header)
c = t[t.region == "Brazil"].copy()
c["date"] = pd.to_datetime(c[["year", "month"]].assign(day=1))
ax = c.plot(x="date", y=["pred", "viirs_obs"], figsize=(12, 4)); ax.set_title("Brazil, monthly"); plt.show()

In [ ]:
# per-cell results stay on Drive (cells_<year>.parquet, ~50 MB each); download the small totals table
files.download(f"{OUT}/monthly_totals.csv")